<a href="https://colab.research.google.com/github/banshitarout16/T5-Text-Summarizer/blob/main/summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [5]:
from google.colab import files

uploaded = files.upload()

Saving samsum-test.csv to samsum-test.csv
Saving samsum-train.csv to samsum-train.csv
Saving samsum-validation.csv to samsum-validation.csv


In [6]:
!pip install transformers
!pip install "transformers[torch]"

In [7]:
import pandas as pd
import re
import torch
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [8]:
train_data = pd.read_csv("samsum-train.csv")
validation_data = pd.read_csv("samsum-validation.csv")

In [9]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [10]:
validation_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [11]:
## Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)
train_data.shape

(4000, 3)

In [12]:
validation_data.shape

(500, 3)

In [13]:
# data Preprocessing ( remove HTML tags, unnecessary space & \r\n lines)

def clean_data(text):
    text=re.sub(r"\r\n", " ", text)
    text=re.sub(r"\s+", " ", text)
    text=re.sub(r"<.*?>", " ", text)
    text = text.strip().lower()
    return text

# apply clean_data to train & val data

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [14]:
validation_data.head(10)

,id,dialogue,summary
0,13680857,"edd: wow, did you hear that they're transferri...",rose and edd will be transferred to a new depa...
1,13716124,"tom: where is the ""sala del capitolo"" kevin: i...","""sala del capitolo"" tom is looking for is in t..."
2,13864418,patricia: the rowing practice is cancelled! ka...,the rowing practice is cancelled. a few member...
3,13729340,"tom: u ok? alex: yeah, pretty good. u? tom: ...",tom and alex had fun last night. they drank a ...
4,13818813,"patricia: hello, here's the fair-trade brand i...",patricia recommends a fair-trade brand she tal...
5,13729102,mark: what time is the breakfast? susanne: 8-1...,susanne will have breakfast at 8 am.
6,13680722,"george: ben, are you going to our choir rehear...",ben is sick and won't come to the choir rehear...
7,13730875,derek: hey derek: yo?? derek: ??? danny: let m...,danny would like to be left to sleep.
8,13730187,iza: monica: omg monica: yesssssss!!! iza: i...,iza has good news.
9,13865465,alice: did you know that amy had an abortion? ...,amy had an abortion.


### *Removed all the HTML tags, unnecessary space & \r\n lines*

In [15]:
# Embedding(tokenizers)

tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [16]:
#raw data -> tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True) #input value
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) #Output value

    inputs["labels"] = targets["input_ids"] # token ids- add to input as labels
    return inputs

In [17]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
validation_dataset = validation_data.apply(tokenize, axis=1).tolist()

In [18]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
len(train_dataset[0]["input_ids"])

512

---
## headup:-
total token = 512

1 => EOs (end of sequals)

0 => padding

above embedded data has 3 major things:-
- input_ids = dialogues converted into token ids
- attention_mask = it shows where the [validid=1, invalid id=0] (ps-if there is token it shows->1, if no token(padding) it refelects->0
- labels = target(summary converted into token )

In [20]:
# working with Model

model = T5ForConditionalGeneration.from_pretrained("t5-small") #use for generational task

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [21]:
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print("device:", device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [22]:
# defining training args

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay=0.01,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [23]:
# defining Trainer - the trainer class provides an API for features training in PyTorch

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

In [24]:
# training

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.649333,0.381290
2,0.397030,0.359758
3,0.374020,0.354646
4,0.361330,0.350265
5,0.355287,0.349108
6,0.350913,0.348678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9146521606445313, metrics={'train_runtime': 1302.6518, 'train_samples_per_second': 18.424, 'train_steps_per_second': 2.303, 'total_flos': 3248203235328000.0, 'train_loss': 0.9146521606445313, 'epoch': 6.0})

In [25]:
# model load -> finetune -> model save

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
save_path = "/content/drive/MyDrive/T5-Text-Summarizer/saved_summary_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/T5-Text-Summarizer/saved_summary_model


In [34]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [35]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [36]:
# test logic - summarization
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  # s-1 tokenize
  inputs = tokenizer(
       dialogue,
       padding = "max_length",
       max_length=512,
       truncation=True,
       return_tensors="pt"
        )
  inputs = {key: value.to(device) for key, value in inputs.items()}

  # s-2 summary generate -> token
  model.to(device)
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # token ids - decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)

  return summary

In [37]:
test_dialogue = """
Reporter: Good evening, Dr. Sharma. Artificial intelligence seems to be everywhere right now. From chatbots to self-driving cars, AI has become part of everyday life. But what exactly is AI?

Expert: Good evening. At its simplest, artificial intelligence is the field of creating computer systems that can perform tasks that normally require human intelligence. These tasks include understanding language, recognizing images, making predictions, solving problems, learning from data, and generating content.

Reporter: When we talk about ChatGPT, Gemini, and other AI assistants, are we talking about artificial intelligence?

Expert: Exactly. These systems are examples of generative AI because they can generate new content such as text, images, audio, video, and code.

Reporter: People often say that large language models know everything. Is that really true?

Expert: Not exactly. An LLM, or Large Language Model, is trained on enormous amounts of text and learns patterns in language. It generates responses based on those learned patterns and the context it receives. It does not understand the world in exactly the same way a human does.

Reporter: Then why do AI systems sometimes give answers that sound completely confident even when they are wrong?

Expert: That is one of the important limitations of modern AI. A model can produce fluent and convincing text without the information necessarily being correct. This is commonly called an AI hallucination. The system may invent a fact, provide an incorrect explanation, or even create a citation that does not exist.

Reporter: Does that mean people should not trust AI?

Expert: People should use AI carefully rather than blindly trusting it. For everyday tasks, AI can be extremely useful. However, information involving medicine, law, finance, safety, or other important decisions should be verified using reliable sources.

Reporter: Another word we hear frequently is Transformer. What exactly is a Transformer?

Expert: A Transformer is a neural network architecture that became extremely important in natural language processing. One of its key ideas is attention. Attention allows the model to consider relationships between different words or tokens in a sequence.

Reporter: Can you explain attention in a simple way?

Expert: Imagine reading a long sentence. When you are trying to understand one particular word, some other words are more relevant than others. Attention allows the model to assign different importance to different parts of the input. This helps it understand relationships and context.

Reporter: Is this the technology described in the famous Attention Is All You Need paper?

Expert: Yes. The 2017 Attention Is All You Need paper introduced the Transformer architecture. It had a huge impact on natural language processing and became an important foundation for many modern language models.

Reporter: Is a Transformer the same thing as an LLM?

Expert: No. A Transformer is an architecture, while an LLM is a large language model. Many modern LLMs use Transformer-based architectures, but the terms mean different things.

Reporter: What about T5? Where does T5 fit into this?

Expert: T5 stands for Text-To-Text Transfer Transformer. It is a Transformer-based language model that treats many NLP tasks as text-to-text problems. For example, an input sentence can be translated into another language, a question can be converted into an answer, or a dialogue can be converted into a summary.

Reporter: So T5 can be used for text summarization?

Expert: Yes. T5 can be fine-tuned on a summarization dataset. During training, the model receives a longer piece of text as input and learns to generate a shorter summary that captures the important information.

Reporter: What would be a practical example of that?

Expert: Imagine a company receiving thousands of customer-service conversations every day. Employees would have to read each conversation to understand what happened. A summarization model could generate a short summary for each conversation, allowing employees to understand the situation much faster.

Reporter: That sounds useful, but it also raises concerns about jobs. Will AI replace human workers?

Expert: AI will certainly change many jobs, but the situation is more complicated than simply saying that AI will replace everyone. Some repetitive tasks can be automated, while other jobs may change because workers will use AI as a tool.

Reporter: Can you give us an example?

Expert: Software development is a good example. AI tools can generate code, explain errors, suggest improvements, and write documentation. However, developers still need to understand the requirements, evaluate the generated code, design systems, and make important decisions.

Reporter: What about students? Is AI helping students or making them dependent on technology?

Expert: It can do either depending on how it is used. If students ask AI to complete every assignment without understanding the material, they may learn less. But if they use AI as a tutor to explain difficult concepts, generate practice questions, or provide hints, it can become a powerful learning tool.

Reporter: So students should use AI as a learning assistant rather than simply copying its answers?

Expert: Exactly. The goal should be understanding, not just obtaining an answer.

Reporter: AI is also being used to generate images, videos, and voices. Does that create a serious misinformation problem?

Expert: Yes. Generative AI makes it easier and cheaper to create convincing fake content. Deepfake videos and artificial voices can be used to impersonate people or spread false information. Even simple AI-generated text can be used to create misleading content at a large scale.

Reporter: How can ordinary people protect themselves from misinformation?

Expert: People should slow down before sharing surprising or emotional content. They should check the original source, compare information with reliable independent sources, and be especially careful when a message creates urgency or asks for money or personal information.

Reporter: Privacy is another concern. People often paste personal information into AI tools. Is that safe?

Expert: People should understand how a particular AI service handles their data before entering sensitive information. Passwords, financial information, confidential company documents, and private personal information should not be casually entered into AI systems.

Reporter: Should businesses adopt AI as quickly as possible to remain competitive?

Expert: Businesses should first identify the problem they are trying to solve. AI should be adopted because it provides a useful solution, not simply because it is fashionable. Companies should consider accuracy, cost, privacy, security, and how they will measure the results.

Reporter: What about creativity? Some artists and writers are worried that generative AI will make human creativity unnecessary.

Expert: I don't think human creativity is becoming irrelevant. AI can generate ideas and variations very quickly, but humans bring personal experiences, intentions, emotions, cultural understanding, and judgment. The creative process may change, but that does not mean human creativity disappears.

Reporter: Do you think AI can actually be creative?

Expert: That depends on how we define creativity. AI can generate novel combinations that humans may consider creative. However, whether that is the same as human creativity is still a philosophical and scientific question.

Reporter: Where do you think artificial intelligence is heading in the future?

Expert: We will probably see AI assistants becoming integrated into many everyday tools. Instead of opening a separate chatbot, people may interact with AI directly inside their documents, programming environments, educational platforms, workplaces, and other applications.

Reporter: Will AI become completely autonomous?

Expert: AI systems are already becoming capable of performing multiple steps toward a goal and using external tools. The important question is how much autonomy we should give them and what safeguards should be in place.

Reporter: Could AI eventually become smarter than humans?

Expert: That is difficult to answer because intelligence is not a single ability. AI already outperforms humans in certain specialized tasks, while humans remain highly flexible across many different real-world situations. The future capabilities of AI are uncertain, so we should avoid both extreme optimism and extreme fear.

Reporter: What is the biggest misunderstanding people have about AI?

Expert: People often treat AI as either magic or as a human mind inside a computer. It is neither. AI systems are based on mathematics, algorithms, data, neural networks, and enormous amounts of computing power. Understanding that helps people use AI more responsibly.

Reporter: What advice would you give to someone who wants to build a career in AI?

Expert: Learn the fundamentals instead of chasing every new AI tool. Learn Python, mathematics, statistics, machine learning, neural networks, NLP, Transformers, databases, and software engineering. Tools change quickly, but strong fundamentals remain valuable.

Reporter: Where does NLP fit into all of this?

Expert: NLP stands for Natural Language Processing. It is the field concerned with enabling computers to process and work with human language. Transformers have dramatically improved what is possible in NLP.

Reporter: And where does generative AI fit?

Expert: Generative AI is a broader category of AI systems that generate new content. This includes text, images, audio, video, and code. LLMs are one of the most important types of generative AI for language-based applications.

Reporter: So if we put all these concepts together, how would you explain their relationship?

Expert: NLP is the broader field. Transformers are an important neural network architecture. LLMs are large language models, many of which use Transformers. Generative AI refers to systems that can create new content. T5 is a Transformer-based language model that can be fine-tuned for tasks such as summarization.

Reporter: Finally, are you optimistic about the future of AI?

Expert: I am cautiously optimistic. AI can help people learn, create, analyze information, automate repetitive work, and solve difficult problems. But technology alone does not guarantee good outcomes. The way we design, deploy, regulate, and use AI matters.

Reporter: So the future is not simply humans versus AI?

Expert: No. A more useful question is what humans can accomplish with AI while maintaining human responsibility, judgment, and oversight.

Reporter: Dr. Sharma, thank you for joining us today.

Expert: Thank you. It was a pleasure.

Reporter: And to our viewers, one thing is clear: artificial intelligence is no longer just a futuristic technology. It is becoming part of how we work, learn, communicate, and make decisions. Understanding both its capabilities and its limitations will be increasingly important."""


summary = summarize_dialogue(test_dialogue)

print("SUMMARY:")
print(summary)

SUMMARY:
ai is a neural network architecture that became extremely important in natural language processing. ai systems can perform tasks that normally require human intelligence. they can generate new content such as text, images, audio, video and code.
